# ⚡ 第7周-Day6：实战搭建完整数字员工原型

> 把前5天的知识串起来：身份(SOUL) → 记忆(Memory) → 工作流(Workflow) → 多Agent → 评估(QA)
> 今天动手搭建一个**完整可运行**的数字员工原型！

In [ ]:
# 配置 matplotlib 中文显示
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")


## 实验1：整合五模块 — 完整原型类

In [ ]:
from dataclasses import dataclass, field
from typing import List
from collections import defaultdict, deque
import time, random

@dataclass
class SoulConfig:
    role: str = "智能客服专员"
    rules: List[str] = field(default_factory=lambda: ["只回答业务相关问题", "不编造数据"])
    output_format: str = "markdown"

@dataclass
class MemoryStore:
    working: list = field(default_factory=list)

    def add_working(self, item):
        if len(self.working) > 5: self.working.pop(0)
        self.working.append(item)

class WorkflowEngine:
    def __init__(self):
        self.tasks = {}
        self.edges = defaultdict(list)

    def add_task(self, name, duration):
        self.tasks[name] = duration

    def add_edge(self, before, after):
        self.edges[before].append(after)

    def run(self):
        in_deg = {n: 0 for n in self.tasks}
        for s, ds in self.edges.items():
            for d in ds:
                if d in in_deg:
                    in_deg[d] += 1
        queue = deque([n for n, d in in_deg.items() if d == 0])
        log = []
        while queue:
            node = queue.popleft()
            dur = self.tasks[node]
            log.append((node, dur))
            for nb in self.edges[node]:
                if nb in in_deg:
                    in_deg[nb] -= 1
                    if in_deg[nb] == 0:
                        queue.append(nb)
        return log

class MultiAgentOrchestrator:
    def __init__(self):
        self.agents = {}
    def add_agent(self, name, skill):
        self.agents[name] = skill
    def dispatch(self, task):
        results = []
        for name, skill in self.agents.items():
            score = max(0, min(10, random.gauss(skill, 1)))
            results.append((name, score))
        return sorted(results, key=lambda x: -x[1])[0]

class QualityEvaluator:
    def evaluate(self):
        return {k: random.uniform(7, 10) for k in ["准确性","完整性","及时性","合规性"]}

print("✅ 五大模块定义完成")

## 实验2：端到端执行模拟

In [ ]:
soul = SoulConfig()
memory = MemoryStore()
memory.working = ["用户咨询退款政策", "用户订单号: ORD-20260818"]

wf = WorkflowEngine()
task_list = [("解析意图",0.2), ("检索知识",0.5), ("生成回复",0.8), ("合规检查",0.3), ("发送回复",0.1)]
for name, dur in task_list:
    wf.add_task(name, dur)
wf.add_edge("解析意图", "检索知识")
wf.add_edge("检索知识", "生成回复")
wf.add_edge("生成回复", "合规检查")
wf.add_edge("合规检查", "发送回复")

orch = MultiAgentOrchestrator()
orch.add_agent("意图识别Agent", 8.5)
orch.add_agent("知识检索Agent", 9.0)
orch.add_agent("回复生成Agent", 7.5)
orch.add_agent("合规审核Agent", 8.0)

qa = QualityEvaluator()

print("🚀 开始执行任务：处理用户退款咨询")
print("-" * 50)

for step, dur in wf.run():
    best = orch.dispatch(step)
    msg = "  [%s] 耗时%ss → 分配给 %s (得分:%.1f)" % (step, dur, best[0], best[1])
    print(msg)

print("-" * 50)
scores = qa.evaluate()
print("📊 质量评估:")
for k, v in scores.items():
    print("  %s: %.1f/10" % (k, v))
avg = np.mean(list(scores.values()))
print("")
print("✅ 平均分: %.1f/10" % avg)

## 实验3：原型架构全景图

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

modules = ["SOUL\n(身份)", "Memory\n(记忆)", "Workflow\n(工作流)", "MultiAgent\n(协作)", "QA\n(评估)"]
x_pos = [1, 3, 5, 7, 9]
colors = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0", "#E91E63"]

ax.scatter(x_pos, [1]*5, s=3000, c=colors, zorder=5, edgecolors='white', linewidth=2)
for x, label in zip(x_pos, modules):
    ax.text(x, 1, label, ha='center', va='center', fontsize=11, fontweight='bold')

for i in range(len(x_pos)-1):
    ax.annotate("", xy=(x_pos[i+1]-0.7, 1), xytext=(x_pos[i]+0.7, 1),
                arrowprops=dict(arrowstyle="->", color="gray", lw=2))

ax.set_xlim(0, 10.5); ax.set_ylim(-1, 3); ax.axis('off')
ax.set_title("数字员工完整原型架构", fontsize=14)

descs = ["角色+规则", "三层记忆", "DAG执行", "任务分发", "雷达评分"]
for x, desc in zip(x_pos, descs):
    ax.text(x, 0.2, desc, ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.show()